In [ ]:
import pandas as pd
import numpy as np

# ==========================================
# 0. טעינת קבצי הנתונים (EEG ופריטים)
# ==========================================
print("Loading Master EEG data and Items data...")
df_data = pd.read_csv('Master_P600_Data_correct_trial_number.csv')
df_items = pd.read_excel('עותק של REPLICATION_ITEMS.xlsx')

# העמודה ה-8 באקסל (אינדקס 7) היא עמודה H - נורמינג ה-Expected
norming_col = df_items.columns[7]

# ==========================================
# 1. הכנת נתוני ה-EEG - מעבדת BIRM
# ==========================================
print("Processing BIRM EEG data...")
df_birm = df_data[df_data['Lab'] == 'BIRM'].copy()
df_birm['Subject_ID'] = df_birm['Subject'].str.extract(r'(\d+)').astype(int)

# ==========================================
# 2. עיבוד קובץ ה-Log של BIRM
# ==========================================
print("Reading BIRM log file...")
df_log = pd.read_csv('birm_log_files_merged.txt', sep='\t', encoding='utf-16', on_bad_lines='skip')

# ניקוי רווחים מיותרים משמות העמודות (למניעת תקלות)
df_log.columns = df_log.columns.str.strip()
df_log['Subject_ID'] = df_log['Subject'].astype(int)

# חישוב ה-Item Number מתוך 4 הבלוקים האפשריים
df_log['Item Number'] = np.nan
for i in range(1, 5):
    col_name = f'Blocks{i}'
    if col_name not in df_log.columns:
        col_name = f'Blocks{i}.Sample' # Fallback

    if col_name in df_log.columns:
        val = pd.to_numeric(df_log[col_name], errors='coerce')
        mask = val.notna()
        # החישוב: (מספר הבלוק - 1) כפול 20 + הערך בבלוק
        df_log.loc[mask, 'Item Number'] = ((i - 1) * 20) + val[mask]

# איסוף זמני ההופעה מכל הבלוקים (RSVP, RSVP1, RSVP2...)
onset_cols = [c for c in df_log.columns if 'RSVP' in c and 'OnsetTime' in c]
for col in onset_cols:
    df_log[col] = pd.to_numeric(df_log[col], errors='coerce')

if onset_cols:
    df_log['OnsetTime'] = df_log[onset_cols].max(axis=1)
else:
    df_log['OnsetTime'] = np.nan

# סינון אימונים ויצירת ה-Trial הכרונולוגי
df_log_clean = df_log.dropna(subset=['Item Number', 'OnsetTime']).copy()
df_log_clean = df_log_clean.sort_values(by=['Subject_ID', 'OnsetTime'])
df_log_clean['Trial_Computed'] = df_log_clean.groupby('Subject_ID').cumcount() + 1
df_log_mapping = df_log_clean[['Subject_ID', 'Trial_Computed', 'Item Number']]

# ==========================================
# 3. מיזוג הנתונים
# ==========================================
print("Merging data...")
df_merged = pd.merge(
    df_birm,
    df_log_mapping,
    left_on=['Subject_ID', 'Trial'],
    right_on=['Subject_ID', 'Trial_Computed'],
    how='inner'
)

# מיזוג עם טבלת הפריטים
df_final = pd.merge(
    df_merged,
    df_items[['Item Number', norming_col]].rename(columns={norming_col: 'Norming'}),
    on='Item Number',
    how='left'
)

# ==========================================
# 4. ניקוי ושמירה
# ==========================================
df_final = df_final.drop(columns=['Subject_ID', 'Trial_Computed'])

print("\nPreview of final BIRM data:")
print(df_final[['Subject', 'Trial', 'Condition', 'Item Number', 'Norming', 'Electrode', 'Mean_Amplitude_uV']].head(15))

output_filename = 'Final_Analysis_Table_BIRM.csv'
df_final.to_csv(output_filename, index=False)
print(f"\nDone! File saved as {output_filename}")

Loading Master EEG data and Items data...


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


Processing BIRM EEG data...
Reading BIRM log file...
Merging data...

Preview of final BIRM data:
   Subject  Trial    Condition  Item Number  Norming Electrode  \
0   birm01   11.0  Control_202          4.0       93        Pz   
1   birm01   11.0  Control_202          4.0       93        Cz   
2   birm01   11.0  Control_202          4.0       93       CP1   
3   birm01   11.0  Control_202          4.0       93       CP2   
4   birm01   11.0  Control_202          4.0       93        P3   
5   birm01   11.0  Control_202          4.0       93        P4   
6   birm01   11.0  Control_202          4.0       93        C3   
7   birm01   11.0  Control_202          4.0       93        C4   
8   birm01   11.0  Control_202          4.0       93        P7   
9   birm01   11.0  Control_202          4.0       93        P8   
10  birm01   11.0  Control_202          4.0       93        O1   
11  birm01   11.0  Control_202          4.0       93        O2   
12  birm01    3.0  Control_202         11.0 

In [ ]:
import pandas as pd
import numpy as np
import glob
import os

# 1. טעינת הנתונים (שימוש בנתיב מלא אם צריך)
df_data = pd.read_csv('Master_P600_Data_correct_trial_number.csv')
df_items = pd.read_excel('עותק של REPLICATION_ITEMS.xlsx')
norming_exp_col = df_items.columns[7] # עמודה H

def process_logs(lab_name, folder_path):
    print(f"\n--- Processing {lab_name} ---")

    # טעינת נתוני ה-EEG של המעבדה
    df_lab = df_data[df_data['Lab'] == lab_name].copy()
    df_lab['Subject_ID'] = df_lab['Subject'].str.extract(r'(\d+)').astype(int)

    # קריאת כל קבצי הלוג בתיקייה
    log_files = glob.glob(os.path.join(folder_path, '*.log'))

    all_logs = []
    for file in log_files:
        # טעינה עם הפרדה לפי טאבים
        temp_df = pd.read_csv(file, sep='\t', skiprows=3, on_bad_lines='skip')
        temp_df.columns = temp_df.columns.str.strip()

        # חילוץ מזהה נבדק
        temp_df['Subject_ID'] = temp_df['Subject'].astype(str).str.extract(r'(\d+)').astype(float)

        # סינון לשורות גירוי (Picture)
        if 'Event Type' in temp_df.columns:
            temp_df = temp_df[temp_df['Event Type'] == 'Picture']
        all_logs.append(temp_df)

    if all_logs:
        df_log = pd.concat(all_logs, ignore_index=True)
        df_log = df_log.dropna(subset=['Subject_ID'])
        df_log['Subject_ID'] = df_log['Subject_ID'].astype(int)

        # חילוץ נתונים
        df_log['Item Number'] = pd.to_numeric(df_log['Itemnumber(str)'], errors='coerce')
        df_log['Trial_Computed'] = pd.to_numeric(df_log['Trialnumber(str)'], errors='coerce')

        # מיפוי נקי
        df_log_mapping = df_log[['Subject_ID', 'Trial_Computed', 'Item Number']].dropna().drop_duplicates()

        # מיזוג
        df_merged = pd.merge(df_lab, df_log_mapping,
                             left_on=['Subject_ID', 'Trial'],
                             right_on=['Subject_ID', 'Trial_Computed'],
                             how='inner')

        df_final = pd.merge(df_merged,
                            df_items[['Item Number', norming_exp_col]].rename(columns={norming_exp_col: 'Norming'}),
                            on='Item Number', how='left')

        df_final = df_final.drop(columns=['Subject_ID', 'Trial_Computed'])
        df_final.to_csv(f'Final_Analysis_Table_{lab_name}.csv', index=False)
        print(f"Saved Final_Analysis_Table_{lab_name}.csv")

# הרצה עבור המעבדות
process_logs('BRIS', '/content/bris log files')
process_logs('GLAS', '/content/glas log files')

/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)



--- Processing BRIS ---
Saved Final_Analysis_Table_BRIS.csv

--- Processing GLAS ---
Saved Final_Analysis_Table_GLAS.csv


In [ ]:
import pandas as pd
import numpy as np
import glob
import os

# 1. טעינת הנתונים (שימוש בנתיב מלא אם צריך)
df_data = pd.read_csv('Master_P600_Data_correct_trial_number.csv')
df_items = pd.read_excel('עותק של REPLICATION_ITEMS.xlsx')
norming_col = df_items.columns[7] # עמודה H (הותאם השם לנוחות)

def parse_eprime_txt(filepath):
    try:
        with open(filepath, 'r', encoding='utf-16') as f:
            lines = f.readlines()
    except UnicodeError:
        with open(filepath, 'r', encoding='utf-8') as f:
            lines = f.readlines()

    trials = []
    current_trial = {}
    in_logframe = False
    subject_id = None

    for line in lines:
        line = line.strip()
        if line.startswith('Subject:'):
            try:
                subject_id = int(line.split(':')[1].strip())
            except ValueError:
                pass
        elif line == '*** LogFrame Start ***':
            in_logframe = True
            current_trial = {'Subject_ID': subject_id}
        elif line == '*** LogFrame End ***':
            in_logframe = False
            if any(k.startswith('Blocks') for k in current_trial.keys()):
                trials.append(current_trial)
        elif in_logframe and ':' in line:
            parts = line.split(':', 1)
            current_trial[parts[0].strip()] = parts[1].strip()
    return pd.DataFrame(trials)

def process_eprime_logs(lab_name, file_pattern):
    print(f"\n--- Processing {lab_name} ---")
    df_lab = df_data[df_data['Lab'] == lab_name].copy()
    if df_lab.empty:
        print(f"No {lab_name} data found in Master.")
        return

    df_lab['Subject_ID'] = df_lab['Subject'].str.extract(r'(\d+)').astype(int)

    txt_files = glob.glob(file_pattern)
    if not txt_files:
        print(f"Warning: No {lab_name} log files found using pattern {file_pattern}!")
        return

    all_logs = []
    for file in txt_files:
        df_temp = parse_eprime_txt(file)
        if not df_temp.empty:
            all_logs.append(df_temp)

    if all_logs:
        df_log = pd.concat(all_logs, ignore_index=True)

        # חישוב ה-Item Number לפי הבלוקים (החלק המעודכן)
        df_log['Item Number'] = np.nan
        for i in range(1, 5):
            col = f'Blocks{i}'
            if col in df_log.columns:
                val = pd.to_numeric(df_log[col], errors='coerce')
                mask = val.notna()
                df_log.loc[mask, 'Item Number'] = ((i - 1) * 20) + val[mask]

        # זמני הופעה
        onset_cols = [c for c in ['RSVP.OnsetTime', 'RSVP1.OnsetTime', 'RSVP2.OnsetTime', 'RSVP3.OnsetTime'] if c in df_log.columns]
        for col in onset_cols:
            df_log[col] = pd.to_numeric(df_log[col], errors='coerce')
        df_log['OnsetTime'] = df_log[onset_cols].max(axis=1)

        df_log_clean = df_log.dropna(subset=['Item Number', 'OnsetTime']).copy()
        df_log_clean = df_log_clean.sort_values(['Subject_ID', 'OnsetTime'])
        df_log_clean['Trial_Computed'] = df_log_clean.groupby('Subject_ID').cumcount() + 1

        df_log_mapping = df_log_clean[['Subject_ID', 'Trial_Computed', 'Item Number']]

        df_merged = pd.merge(df_lab, df_log_mapping, left_on=['Subject_ID', 'Trial'], right_on=['Subject_ID', 'Trial_Computed'], how='inner')
        df_final = pd.merge(df_merged, df_items[['Item Number', norming_col]].rename(columns={norming_col: 'Norming'}), on='Item Number', how='left')
        df_final = df_final.drop(columns=['Subject_ID', 'Trial_Computed'])
        df_final.to_csv(f'Final_Analysis_Table_{lab_name}.csv', index=False)
        print(f"{lab_name} saved successfully.")

# ==========================================
# Run all processes
# ==========================================
process_eprime_logs('EDIN', '/content/edin log files/*.txt')
process_eprime_logs('KENT', '/content/kent log files/*.txt')

/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)



--- Processing EDIN ---
EDIN saved successfully.

--- Processing KENT ---
KENT saved successfully.


In [ ]:
import pandas as pd
import numpy as np
import glob
import os

# ==========================================
# 0. טעינת הנתונים
# ==========================================
print("Loading Master EEG data and Items data...")
df_data = pd.read_csv('Master_P600_Data_correct_trial_number.csv')
df_items = pd.read_excel('עותק של REPLICATION_ITEMS.xlsx')
norming_exp_col = df_items.columns[7] # עמודה H

# ==========================================
# 1. פונקציית פענוח דינמית לקובצי Log
# ==========================================
def parse_log_file(filepath):
    # מציאת השורה שמתחילה בכותרת 'Subject'
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()

    header_idx = -1
    for i, line in enumerate(lines):
        if line.startswith('Subject\t'): # מחפשים את שורת הכותרת
            header_idx = i
            break

    if header_idx == -1:
        print(f"Could not find header in {filepath}")
        return pd.DataFrame()

    # קריאת הקובץ החל משורת הכותרת שנמצאה
    df = pd.read_csv(filepath, sep='\t', skiprows=header_idx, on_bad_lines='skip')
    df.columns = df.columns.str.strip()
    return df

# ==========================================
# 2. עיבוד נתוני LOND
# ==========================================
def process_lond_logs(lab_name, folder_path):
    print(f"\n--- Processing {lab_name} ---")
    df_lab = df_data[df_data['Lab'] == lab_name].copy()
    if df_lab.empty:
        print(f"No {lab_name} data found in Master.")
        return

    df_lab['Subject_ID'] = df_lab['Subject'].str.extract(r'(\d+)').astype(int)

    log_files = glob.glob(os.path.join(folder_path, '*.log'))
    all_logs = []

    for file in log_files:
        temp_df = parse_log_file(file)
        if not temp_df.empty:
            # חילוץ מזהה נבדק מהעמודה (לדוגמה 'subj005_002' -> 5)
            # משתמשים ב-regex כדי למצוא מספרים אחרי 'subj'
            temp_df['Subject_ID'] = temp_df['Subject'].astype(str).str.extract(r'(\d+)').astype(float)

            # סינון לשורות גירוי (Picture)
            if 'Event Type' in temp_df.columns:
                temp_df = temp_df[temp_df['Event Type'] == 'Picture']
            all_logs.append(temp_df)

    if all_logs:
        df_log = pd.concat(all_logs, ignore_index=True)
        df_log = df_log.dropna(subset=['Subject_ID'])
        df_log['Subject_ID'] = df_log['Subject_ID'].astype(int)

        # חילוץ Item Number ו-Trial
        df_log['Item Number'] = pd.to_numeric(df_log['Itemnumber(str)'], errors='coerce')
        df_log['Trial_Computed'] = pd.to_numeric(df_log['Trialnumber(str)'], errors='coerce')

        # מיפוי נקי (מסיר כפילויות של מילים בתוך משפט)
        df_log_mapping = df_log[['Subject_ID', 'Trial_Computed', 'Item Number']].dropna().drop_duplicates()

        # מיזוג
        df_merged = pd.merge(df_lab, df_log_mapping,
                             left_on=['Subject_ID', 'Trial'],
                             right_on=['Subject_ID', 'Trial_Computed'],
                             how='inner')

        df_final = pd.merge(df_merged,
                            df_items[['Item Number', norming_exp_col]].rename(columns={norming_exp_col: 'Norming'}),
                            on='Item Number', how='left')

        df_final = df_final.drop(columns=['Subject_ID', 'Trial_Computed'])
        df_final.to_csv(f'Final_Analysis_Table_{lab_name}.csv', index=False)
        print(f"Saved Final_Analysis_Table_{lab_name}.csv with {len(df_final)} rows.")
    else:
        print(f"No valid data found in logs for {lab_name}.")

# הרצה (עדכן את הנתיב אם התיקייה לא נמצאת ב-/content/ באופן ישיר)
process_lond_logs('LOND', '/content/lond log files/')

Loading Master EEG data and Items data...


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)



--- Processing LOND ---
Saved Final_Analysis_Table_LOND.csv with 17580 rows.


In [ ]:
import pandas as pd
import numpy as np
import glob
import os

# ==========================================
# 0. טעינת קבצי הנתונים המרכזיים
# ==========================================
print("Loading Master EEG data and Items data...")
df_data = pd.read_csv('Master_P600_Data_correct_trial_number.csv')
df_items = pd.read_excel('עותק של REPLICATION_ITEMS.xlsx')

# עמודה H היא ה-Norming המבוקש (אינדקס 7)
norming_col = df_items.columns[7]

# ==========================================
# 1. פונקציית פענוח דינמית לקבצי Log
# ==========================================
def parse_log_file(filepath):
    # פתיחת הקובץ וחיפוש השורה שמתחילה ב-Subject כדי לזהות את הכותרת
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()

    header_idx = -1
    for i, line in enumerate(lines):
        if line.startswith('Subject\t'):
            header_idx = i
            break

    if header_idx == -1:
        print(f"Error: Could not find header in {filepath}")
        return pd.DataFrame()

    # קריאת הקובץ החל משורת הכותרת שנמצאה
    df = pd.read_csv(filepath, sep='\t', skiprows=header_idx, on_bad_lines='skip')
    df.columns = df.columns.str.strip()
    return df

# ==========================================
# 2. פונקציית עיבוד מאוחדת ל-OXFO ו-YORK
# ==========================================
def process_lab_logs(lab_name, folder_path):
    print(f"\n--- Processing {lab_name} ---")

    # טעינת נתוני המעבדה מהמאסטר
    df_lab = df_data[df_data['Lab'] == lab_name].copy()
    if df_lab.empty:
        print(f"No {lab_name} data found in Master.")
        return

    df_lab['Subject_ID'] = df_lab['Subject'].str.extract(r'(\d+)').astype(int)

    # חיפוש קבצי ה-log בתיקייה שצוינה
    search_path = os.path.join(folder_path, '*.log')
    log_files = glob.glob(search_path)

    if not log_files:
        print(f"Warning: No log files found in {folder_path}!")
        return

    all_logs = []
    for file in log_files:
        temp_df = parse_log_file(file)
        if not temp_df.empty:
            # חילוץ מזהה נבדק מהעמודה
            temp_df['Subject_ID'] = temp_df['Subject'].astype(str).str.extract(r'(\d+)').astype(int)

            # סינון לשורות גירוי (Picture)
            if 'Event Type' in temp_df.columns:
                temp_df = temp_df[temp_df['Event Type'] == 'Picture']
            all_logs.append(temp_df)

    if all_logs:
        df_log = pd.concat(all_logs, ignore_index=True)

        # חילוץ Item Number ו-Trial
        df_log['Item Number'] = pd.to_numeric(df_log['Itemnumber(str)'], errors='coerce')
        df_log['Trial_Computed'] = pd.to_numeric(df_log['Trialnumber(str)'], errors='coerce')

        # מיפוי נקי (סינון כפילויות למניעת בעיות במיזוג)
        df_log_mapping = df_log[['Subject_ID', 'Trial_Computed', 'Item Number']].dropna().drop_duplicates()

        # מיזוג עם נתוני ה-EEG
        df_merged = pd.merge(df_lab, df_log_mapping,
                             left_on=['Subject_ID', 'Trial'],
                             right_on=['Subject_ID', 'Trial_Computed'],
                             how='inner')

        # מיזוג עם ה-Norming
        df_final = pd.merge(df_merged,
                            df_items[['Item Number', norming_col]].rename(columns={norming_col: 'Norming'}),
                            on='Item Number', how='left')

        # ניקוי עמודות עזר ושמירה
        df_final = df_final.drop(columns=['Subject_ID', 'Trial_Computed'])
        output_name = f'Final_Analysis_Table_{lab_name}.csv'
        df_final.to_csv(output_name, index=False)
        print(f"Saved {output_name} with {len(df_final)} rows.")
    else:
        print(f"No valid data found for {lab_name}.")

# ==========================================
# 3. הרצת העיבוד (עדכן את הנתיבים לפי ה-Content שלך)
# ==========================================
process_lab_logs('OXFO', '/content/oxfo log files/')
process_lab_logs('YORK', '/content/york log files/')

Loading Master EEG data and Items data...

--- Processing OXFO ---


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


Saved Final_Analysis_Table_OXFO.csv with 14988 rows.

--- Processing YORK ---
Saved Final_Analysis_Table_YORK.csv with 19224 rows.


In [ ]:
import pandas as pd
import numpy as np

# ==========================================
# 0. טעינת קבצי הנתונים (EEG ופריטים)
# ==========================================
print("Loading Master EEG data and Items data...")
df_data = pd.read_csv('Master_P600_Data_correct_trial_number.csv')
df_items = pd.read_excel('עותק של REPLICATION_ITEMS.xlsx')

norming_col = df_items.columns[7]
print(f"Using column '{norming_col}' for Norming.")

# ==========================================
# 1. הכנת נתוני ה-EEG - מעבדת STIR
# ==========================================
print("Processing STIR EEG data...")
df_stir = df_data[df_data['Lab'] == 'STIR'].copy()
df_stir['Subject_ID'] = df_stir['Subject'].str.extract(r'(\d+)').astype(int)

# ==========================================
# 2. עיבוד קובץ ה-Log של STIR
# ==========================================
print("Reading STIR log file...")
df_log = pd.read_csv('stir_logs_merged.txt', sep='\t', encoding='utf-16', on_bad_lines='skip')
df_log.columns = df_log.columns.str.strip()
df_log['Subject_ID'] = df_log['Subject'].astype(int)

# חישוב ה-Item Number לפי הבלוקים (מעודכן)
df_log['Item Number'] = np.nan
for i in range(1, 5):
    col_name = f'Blocks{i}'
    if col_name not in df_log.columns:
        col_name = f'Blocks{i}.Sample'

    if col_name in df_log.columns:
        val = pd.to_numeric(df_log[col_name], errors='coerce')
        mask = val.notna()
        df_log.loc[mask, 'Item Number'] = ((i - 1) * 20) + val[mask]

# איסוף זמני הופעה
onset_cols = [c for c in df_log.columns if 'RSVP' in c and 'OnsetTime' in c]
for col in onset_cols:
    df_log[col] = pd.to_numeric(df_log[col], errors='coerce')

if onset_cols:
    df_log['OnsetTime'] = df_log[onset_cols].max(axis=1)
else:
    df_log['OnsetTime'] = np.nan

df_log_clean = df_log.dropna(subset=['Item Number', 'OnsetTime']).copy()
df_log_clean = df_log_clean.sort_values(by=['Subject_ID', 'OnsetTime'])
df_log_clean['Trial_Computed'] = df_log_clean.groupby('Subject_ID').cumcount() + 1
df_log_mapping = df_log_clean[['Subject_ID', 'Trial_Computed', 'Item Number']]

# ==========================================
# 3. מיזוג הנתונים
# ==========================================
print("Merging data...")
df_merged = pd.merge(
    df_stir,
    df_log_mapping,
    left_on=['Subject_ID', 'Trial'],
    right_on=['Subject_ID', 'Trial_Computed'],
    how='inner'
)

df_final = pd.merge(
    df_merged,
    df_items[['Item Number', norming_col]].rename(columns={norming_col: 'Norming'}),
    on='Item Number',
    how='left'
)

# ==========================================
# 4. ניקוי ושמירה
# ==========================================
df_final = df_final.drop(columns=['Subject_ID', 'Trial_Computed'])

print("\nPreview of final STIR data:")
cols_to_print = ['Subject', 'Trial', 'Condition', 'Item Number', 'Norming', 'Electrode', 'Mean_Amplitude_uV']
print(df_final[[c for c in cols_to_print if c in df_final.columns]].head(15))

output_filename = 'Final_Analysis_Table_STIR.csv'
df_final.to_csv(output_filename, index=False)
print(f"\nDone! File saved as {output_filename}")

Loading Master EEG data and Items data...
Using column 'Norming' for Norming.
Processing STIR EEG data...


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


Reading STIR log file...
Merging data...

Preview of final STIR data:
   Subject  Trial      Condition  Item Number  Norming Electrode  \
0   STIR10   12.0  Violation_201          9.0       93        Pz   
1   STIR10   12.0  Violation_201          9.0       93        Cz   
2   STIR10   12.0  Violation_201          9.0       93       CP1   
3   STIR10   12.0  Violation_201          9.0       93       CP2   
4   STIR10   12.0  Violation_201          9.0       93        P3   
5   STIR10   12.0  Violation_201          9.0       93        P4   
6   STIR10   12.0  Violation_201          9.0       93        C3   
7   STIR10   12.0  Violation_201          9.0       93        C4   
8   STIR10   12.0  Violation_201          9.0       93        P7   
9   STIR10   12.0  Violation_201          9.0       93        P8   
10  STIR10   12.0  Violation_201          9.0       93        O1   
11  STIR10   12.0  Violation_201          9.0       93        O2   
12  STIR10   20.0  Violation_201          2.0 